## Use LangChain's built-in tools and understand how tool calling works

In [3]:
# Install (if needed)
# pip install langchain langchain-community wikipedia

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain_core.tools import Tool   # Updated import for LangChain 1.x


# ------------------------------------------------------------
# Step 1: Initialize Wikipedia tool
# ------------------------------------------------------------
wiki = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

wikipedia_tool = Tool(
    name="Wikipedia",
    func=wiki.run,   # callable function for the tool
    description="Useful for fetching facts from Wikipedia"
)


# ------------------------------------------------------------
# Step 2: Simple Calculator tool
# ------------------------------------------------------------
calculator_tool = Tool(
    name="Calculator",
    func=lambda x: eval(x),  # simple math evaluator
    description="Useful for running math expressions like '3+12'"
)


# ------------------------------------------------------------
# Step 3: Call tools directly (no agent)
# ------------------------------------------------------------
query_1 = "Machine learning"
result_1 = wikipedia_tool.run(query_1)

query_2 = "12 * 7 + 5"
result_2 = calculator_tool.run(query_2)


# ------------------------------------------------------------
# Step 4: Print results
# ------------------------------------------------------------
print("\nWikipedia Tool Result:\n")
print(result_1)

print("\nCalculator Tool Result:\n")
print(result_2)



Wikipedia Tool Result:

Page: Machine learning
Summary: Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions. Within a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of statistical algorithms, to surpass many previous machine learning approaches in performance.
ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, agriculture, and medicine. The application of ML to business problems is known as predictive analytics.
Statistics and mathematical optimisation (mathematical programming) methods comprise the foundations of machine learning. Data mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.
From a theor

## Build a simple agent using a prompt template + 2 tools (e.g., calculator + search)

In [6]:
import os
os.environ["OPENAI_API_KEY"] = "sk-"

In [5]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.tools import Tool
from langchain_core.runnables import RunnableLambda, RunnableBranch

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper


# ------------------------------------------------------------
# Step 1: LLM
# ------------------------------------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)


# ------------------------------------------------------------
# Step 2: Tools
# ------------------------------------------------------------
calculator = Tool(
    name="Calculator",
    func=lambda x: str(eval(x)),
    description="Compute math expressions such as '12 * 7 + 5'"
)

wiki_runner = WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())
wikipedia = Tool(
    name="Wikipedia",
    func=wiki_runner.run,
    description="Search Wikipedia for factual information."
)


# ------------------------------------------------------------
# Step 3: Prompt for tool decision
# ------------------------------------------------------------
decision_prompt = PromptTemplate.from_template("""
You are a tool-using agent.

TOOLS:
- Calculator → for math
- Wikipedia → for factual answers

Question: {question}

Decide action using EXACTLY one of:
1. TOOL=Calculator | INPUT=...
2. TOOL=Wikipedia | INPUT=...
3. NO_TOOL

Respond with ONLY one line.
""")

# Runnable: ask LLM what to do
decide_tool = decision_prompt | llm | RunnableLambda(lambda msg: msg.content)


# ------------------------------------------------------------
# Step 4: Tool execution handler
# ------------------------------------------------------------
def run_tool(data):
    decision = data["decision"]

    if decision.startswith("TOOL=Calculator"):
        expr = decision.split("INPUT=")[-1].strip()
        return calculator.run(expr)

    if decision.startswith("TOOL=Wikipedia"):
        query = decision.split("INPUT=")[-1].strip()
        return wikipedia.run(query)

    return None


tool_executor = RunnableLambda(run_tool)


# ------------------------------------------------------------
# Step 5: Fallback LLM (no tool needed)
# ------------------------------------------------------------
direct_answer = PromptTemplate.from_template("""
Answer the following question directly:

{question}
""") | llm | RunnableLambda(lambda msg: msg.content)


# ------------------------------------------------------------
# Step 6: Build the final agent
# ------------------------------------------------------------
agent = (
    # Pass question forward
    {"question": lambda x: x["question"], 
     "decision": decide_tool}
    |
    RunnableBranch(
        (lambda x: x["decision"].startswith("TOOL="), tool_executor),
        direct_answer
    )
)


# ------------------------------------------------------------
# Step 7: TEST
# ------------------------------------------------------------
print("\n--- Wikipedia Query ---")
print(agent.invoke({"question": "Who is the president of France?"}))

print("\n--- Math Query ---")
print(agent.invoke({"question": "12 * 7 + 5"}))

print("\n--- Direct Answer Query ---")
print(agent.invoke({"question": "Explain machine learning in one sentence."}))



--- Wikipedia Query ---
Page: President of France
Summary: The president of France, officially the president of the French Republic (French: Président de la République française, [pʁezidɑ̃ d(ə) la ʁepyblik fʁɑ̃sɛːz]) or president of the Republic (Président de la République), is the executive head of state of France, and the commander-in-chief of the French Armed Forces. As the presidency is the supreme magistracy of the country, the position is the highest office in France. The powers, functions and duties of prior presidential offices, in addition to their relation with the prime minister and government of France, have over time differed with the various constitutional documents since the Second Republic.
The president of France is the ex officio co-prince of Andorra, grand master of the Legion of Honour and of the National Order of Merit, and protector of the Institut de France in Paris. The officeholder is also honorary proto-canon of the Archbasilica of Saint John Lateran in Rome,